> **Verzija s rješenjima — samo za nastavnika.**

# V5: N-gramski jezični modeli

**I051 Računalno jezikoslovlje** - vježbe uz _5. predavanje_.

| | Predavanje P5 | Vježbe |
|---|---|---|
| 1 | n-grami, procjena najveće vjerodostojnosti (MLE) | bigramski model s `Counter`, vjerojatnost rečenice |
| 2 | uzorkovanje | generiranje rečenica |
| 3 | vrednovanje | nepoznate riječi `<unk>`, perpleksija na testnom skupu |
| 4 | izglađivanje | Laplace, dodaj-k, interpolacija |

Model gradimo sami, u nekoliko kratkih funkcija: tako se vidi da je n-gramski model samo **brojanje**.

# 0. Priprema

Bilježnica koristi samo standardnu biblioteku Pythona (`collections`, `math`, `random`), pa ništa ne treba instalirati.

U mapi `data` su rečenice korpusa UD Croatian SET (isti kao u V2): `hr_train.txt` za učenje i `hr_test.txt` za testiranje. Rečenice su već tokenizirane: jedna rečenica po retku, tokeni odvojeni razmakom.

Funkcija `read_sentences` čita datoteku u listu rečenica; svaka rečenica je lista tokena malim slovima, s oznakama `<s>` (početak) i `</s>` (kraj).

In [ ]:
import math
import random
from collections import Counter

def read_sentences(path):
    sentences = []
    for line in open(path, encoding="utf-8"):
        sentences.append(["<s>"] + line.lower().split() + ["</s>"])
    return sentences

train = read_sentences("data/hr_train.txt")
test = read_sentences("data/hr_test.txt")
print(len(train), "rečenica za učenje,", len(test), "za testiranje")
print(train[3])

# 1. Bigramski model

S predavanja: procjena najveće vjerodostojnosti (MLE) bigrama je relativna frekvencija

$$P(w_n \mid w_{n-1}) = \frac{C(w_{n-1}\,w_n)}{C(w_{n-1})}$$

gdje je $C(\cdot)$ broj pojavljivanja u korpusu za učenje. Trebaju nam dva brojača: broj riječi (unigrama) i broj parova susjednih riječi (bigrama). Parove daje `zip(words, words[1:])`.

Najprije mali korpus s predavanja:

In [ ]:
def count_ngrams(sentences):
    unigrams, bigrams = Counter(), Counter()
    for words in sentences:
        unigrams.update(words)                   # svaka riječ
        bigrams.update(zip(words, words[1:]))    # svaki par susjednih riječi
    return unigrams, bigrams

toy = [s.split() for s in ["<s> ja sam Ana </s>",
                           "<s> Ana ja sam </s>",
                           "<s> ja ne volim zelena jaja i šunku </s>"]]
unigrams, bigrams = count_ngrams(toy)
print(unigrams["ja"], bigrams[("ja", "sam")])
bigrams.most_common(3)

Vjerojatnost je jedan redak — upravo formula MLE:

In [ ]:
def p(word, prev):
    return bigrams[(prev, word)] / unigrams[prev]

print(p("sam", "ja"))     # 2/3, kao na slajdu »Primjer: mali korpus«
print(p("Ana", "<s>"))    # 1/3

## 1.1 Model na korpusu SET

Isti kod, samo drugi podaci. Funkcija `p` koristi brojače `unigrams` i `bigrams`, pa nakon novog brojanja računa vjerojatnosti iz SET-a.

In [ ]:
unigrams, bigrams = count_ngrams(train)
print(len(unigrams), "različitih riječi,", len(bigrams), "različitih bigrama")
print(p("vanjskih", "ministar"))
print(p("je", "rekao"))

Koje riječi model očekuje iza zadane riječi? Skupimo sve bigrame koji počinju s `prev` i poredajmo ih po broju pojavljivanja.

In [ ]:
def next_words(prev, n=5):
    following = Counter({word: count for (first, word), count in bigrams.items() if first == prev})
    return [(word, round(count / unigrams[prev], 3)) for word, count in following.most_common(n)]

next_words("ministar")

**Pokušajte:** što slijedi iza *rekao*, *hrvatska* i *<s>* (kojim riječima počinju rečenice)?

In [ ]:
for word in ["rekao", "hrvatska", "<s>"]:
    print(word, next_words(word))

## 1.2 Vjerojatnost rečenice

Vjerojatnost rečenice je umnožak bigramskih vjerojatnosti. Umnožak mnogo malih brojeva brzo postane premalen, pa **zbrajamo logaritme**. Ako je neki bigram nula, cijela vjerojatnost je 0, a njezin logaritam $-\infty$ (`-math.inf`).

In [ ]:
def logprob(sentence):
    words = sentence.split()
    total = 0
    for prev, word in zip(words, words[1:]):
        prob = p(word, prev)
        if prob == 0:
            return -math.inf      # log(0)
        total += math.log(prob)
    return total

s1 = "<s> vlada je rekla da će se to dogoditi . </s>"
s2 = "<s> vlada je rekla da će se to dogoditi </s>"     # bez točke
print(logprob(s1), math.exp(logprob(s1)))
print(logprob(s2))

### Zadatak 1

Za rečenicu `s2` ispišite svaki bigram, njegov broj u korpusu i vjerojatnost. Koji bigram je nula? Zatim isto napravite za rečenicu *&lt;s&gt; ministar vanjskih poslova izjavio je kako . &lt;/s&gt;* — je li nula opravdana?

In [ ]:
for sentence in [s2, "<s> ministar vanjskih poslova izjavio je kako . </s>"]:
    words = sentence.split()
    for prev, word in zip(words, words[1:]):
        print(f"{prev:10} {word:10} {bigrams[(prev, word)]:5} {p(word, prev):.4f}")
    print()
# s2: »dogoditi </s>« = 0 — u korpusu rečenice završavaju točkom, pa je nula opravdana.
# Druga rečenica: »poslova izjavio« i »kako .« su 0 — rečenica je neobična, ali »poslova izjavio« je slučajna nula.

**Za razmisliti:** na predavanju smo pitali je li bolje »Idem u kino sa *sestrom*« ili »sa *šestom*«. Izračunajte `bigrams[("sa", "sestrom")]` i `bigrams[("sa", "šestom")]`. Što bi odlučio ovaj model i zašto? (Pomoć: od čega je sastavljen korpus SET?)

# 2. Generiranje rečenica

**Uzorkovanje**: sljedeću riječ biramo slučajno, ali s vjerojatnošću $P(w \mid \text{prethodna})$. Funkcija `random.choices(lista, weights=težine)` bira element liste s težinama; težine su brojevi bigrama (ne moraju se zbrajati u 1). Generiramo od `<s>` dok ne izvučemo `</s>`.

In [ ]:
def next_word(prev):
    candidates = [word for (first, word) in bigrams if first == prev]
    weights = [bigrams[(prev, word)] for word in candidates]
    return random.choices(candidates, weights=weights)[0]

def generate():
    words = ["<s>"]
    while words[-1] != "</s>":
        words.append(next_word(words[-1]))
    return " ".join(words[1:-1])

random.seed(4)      # isti slučajni brojevi kao na slajdu
for i in range(5):
    print(generate())

Svaki par susjednih riječi postoji u korpusu, ali rečenica u cjelini nema smisla: model vidi samo jednu riječ unatrag.

### Zadatak 2

Dopunite funkciju `generate_from(start)` tako da rečenica počinje zadanim riječima (npr. *ministar vanjskih*), a dalje se generira kao prije. Generirajte po tri rečenice za *ministar vanjskih* i *hrvatska je*.

In [ ]:
def generate_from(start):
    words = ["<s>"] + start.split()
    while words[-1] != "</s>":
        words.append(next_word(words[-1]))
    return " ".join(words[1:-1])

random.seed(1)
for start in ["ministar vanjskih", "hrvatska je"]:
    for i in range(3):
        print(generate_from(start))

# 3. Vrednovanje: perpleksija

## 3.1 Nepoznate riječi

Perpleksiju računamo na testnom skupu, a u njemu ima riječi kojih nema u skupu za učenje. Njihova vjerojatnost bila bi 0. Zato, kao na predavanju, napravimo **zatvoren rječnik**: riječi koje se u skupu za učenje pojavljuju barem dvaput; sve ostale, u oba skupa, zamijenimo oznakom `<unk>`.

In [ ]:
word_counts = Counter(word for words in train for word in words)
vocab = {word for word, count in word_counts.items() if count >= 2}

def replace_rare(sentences):
    return [[word if word in vocab else "<unk>" for word in words] for words in sentences]

train_u = replace_rare(train)
test_u = replace_rare(test)
print(test[3])
print(test_u[3])

Brojimo ponovno, sada s `<unk>`. Trebaju nam i dvije veličine:

* `V` — veličina rječnika: broj različitih riječi koje model može predvidjeti (`<s>` se nikad ne predviđa, pa ga ne brojimo),
* `N` — broj tokena koje predviđamo (svi osim `<s>`).

In [ ]:
unigrams, bigrams = count_ngrams(train_u)
V = len(unigrams) - 1
N = sum(unigrams.values()) - unigrams["<s>"]
print("V =", V, " N =", N)

## 3.2 Perpleksija

S predavanja: perpleksija je inverz vjerojatnosti testnog skupa, normaliziran brojem predviđanja $n$:

$$PP(W) = P(w_1 \ldots w_n)^{-1/n} = \exp\Big(-\frac{1}{n}\sum \log P(w_i \mid w_{i-1})\Big)$$

Funkcija `perplexity` prima rečenice i **funkciju** `prob(word, prev)` koja daje vjerojatnost. Tako istom funkcijom vrednujemo različite modele.

In [ ]:
def perplexity(sentences, prob):
    total, n = 0, 0
    for words in sentences:
        for prev, word in zip(words, words[1:]):
            q = prob(word, prev)
            if q == 0:
                return math.inf          # jedna nula -> beskonačna perpleksija
            total += math.log(q)
            n += 1
    return math.exp(-total / n)

def p_unigram(word, prev):               # unigramski model: prethodna riječ se ne gleda
    return unigrams[word] / N

def p_bigram(word, prev):
    if unigrams[prev] == 0:              # povijest koju nismo vidjeli
        return 0.0
    return bigrams[(prev, word)] / unigrams[prev]

print("unigram:", perplexity(test_u, p_unigram))
print("bigram: ", perplexity(test_u, p_bigram))

Bigramski model ima beskonačnu perpleksiju: dovoljan je jedan testni bigram koji se ne pojavljuje u skupu za učenje.

**Pokušajte:** izračunajte perpleksiju bigramskog modela na **skupu za učenje** (`train_u`). Zašto je tako mala? Što to govori o vrednovanju na podacima za učenje?

In [ ]:
print(perplexity(train_u, p_bigram))
# ~31: model je te bigrame izbrojao, pa ih »zna napamet« (prenaučenost). Zato vrednujemo na testnom skupu.

Koliko testnih bigrama model nikad nije vidio?

In [ ]:
test_pairs = [pair for words in test_u for pair in zip(words, words[1:])]
unseen = sum(1 for pair in test_pairs if bigrams[pair] == 0)
print(f"{unseen} od {len(test_pairs)} testnih bigrama ({unseen / len(test_pairs):.1%}) nema u skupu za učenje")

# 4. Izglađivanje

## 4.1 Laplaceovo izglađivanje

Svakom broju bigrama dodamo 1, a nazivniku $V$:

$$P_{\text{Laplace}}(w_n \mid w_{n-1}) = \frac{C(w_{n-1}\,w_n) + 1}{C(w_{n-1}) + V}$$

In [ ]:
def p_laplace(word, prev):
    return (bigrams[(prev, word)] + 1) / (unigrams[prev] + V)

print("P(je | rekao):", p_bigram("je", "rekao"), "->", p_laplace("je", "rekao"))
print("Laplace:", perplexity(test_u, p_laplace))

Vjerojatnost viđenog bigrama *rekao je* pala je s 0,71 na 0,008 (na slajdu, bez `<unk>`, rječnik je veći pa je pad još jači). Nula je nestala, ali perpleksija je **lošija od unigramskog modela**: jedinice za sve neviđene bigrame odnesu previše vjerojatnosti onima koje smo vidjeli.

## 4.2 Dodaj-k i razvojni skup

Umjesto 1 dodajemo manji broj $k$. Broj $k$ je **hiperparametar**: biramo ga na **razvojnom skupu**, a ne na testnom. Za razvojni skup odvojimo zadnjih 10 % rečenica za učenje i privremeno brojimo samo na prvih 90 %.

`lambda word, prev: p_addk(word, prev, k)` je kratko napisana funkcija od dva argumenta: `perplexity` je poziva kao i `p_laplace`, a `k` uzima iz petlje.

In [ ]:
def p_addk(word, prev, k):
    return (bigrams[(prev, word)] + k) / (unigrams[prev] + k * V)

split = int(len(train_u) * 0.9)
dev = train_u[split:]                                  # razvojni skup
unigrams, bigrams = count_ngrams(train_u[:split])      # privremeno: bez razvojnog dijela
N = sum(unigrams.values()) - unigrams["<s>"]

for k in [1, 0.1, 0.01, 0.001]:
    print(k, round(perplexity(dev, lambda word, prev: p_addk(word, prev, k)), 1))

Najbolji je $k = 0{,}01$. Tek sada, s izabranim $k$, brojimo na cijelom skupu za učenje i jednom mjerimo na testu:

In [ ]:
unigrams, bigrams = count_ngrams(train_u)
N = sum(unigrams.values()) - unigrams["<s>"]
print("dodaj-k:", perplexity(test_u, lambda word, prev: p_addk(word, prev, 0.01)))

## 4.3 Interpolacija

Miješamo unigramski i bigramski model s težinom $\lambda$:

$$\hat P(w_n \mid w_{n-1}) = \lambda\, P(w_n) + (1 - \lambda)\, P(w_n \mid w_{n-1})$$

Kad bigrama nema, ostaje unigramski dio. Da ni on ne bi bio nula (na razvojnom skupu mogu se pojaviti riječi koje nisu u prvih 90 %), unigram izglađujemo Laplaceom.

In [ ]:
def p_unigram_laplace(word, prev):
    return (unigrams[word] + 1) / (N + V)

def p_interp(word, prev, lam):
    return lam * p_unigram_laplace(word, prev) + (1 - lam) * p_bigram(word, prev)

print("interpolacija, λ = 0.5:", perplexity(test_u, lambda word, prev: p_interp(word, prev, 0.5)))

### Zadatak 3

Težinu $\lambda$ smo izabrali napamet. Izaberite je ispravno: kao za $k$, brojite na prvih 90 % skupa za učenje i za $\lambda \in \{0{,}1;\ 0{,}2;\ \ldots;\ 0{,}9\}$ ispišite perpleksiju na razvojnom skupu. Zatim za najbolju $\lambda$ izračunajte perpleksiju na testnom skupu (brojevi iz cijelog skupa za učenje).

In [ ]:
unigrams, bigrams = count_ngrams(train_u[:split])
N = sum(unigrams.values()) - unigrams["<s>"]
for lam in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    print(lam, round(perplexity(dev, lambda word, prev: p_interp(word, prev, lam)), 1))

unigrams, bigrams = count_ngrams(train_u)
N = sum(unigrams.values()) - unigrams["<s>"]
best = 0.6
print("test:", perplexity(test_u, lambda word, prev: p_interp(word, prev, best)))

## 4.4 Usporedba

Svi modeli imaju isti rječnik, pa su perpleksije usporedive:

In [ ]:
models = [("unigram", p_unigram),
          ("bigram (MLE)", p_bigram),
          ("bigram, Laplace", p_laplace),
          ("bigram, dodaj-k (k = 0.01)", lambda word, prev: p_addk(word, prev, 0.01)),
          ("interpolacija (λ = 0.6)", lambda word, prev: p_interp(word, prev, 0.6))]
for name, prob in models:
    print(f"{name:28} {perplexity(test_u, prob):8.1f}")

**Za razmisliti:** na predavanju je trigramska interpolacija dala 191,3, a ovdje bigramska daje oko 200. Zašto trigrami na korpusu od 6914 rečenica pomažu tako malo? Što bi se dogodilo s korpusom sto puta većim?